# Model 3 — Crop Yield Predictor (Kerala)

This notebook trains a **tabular yield model** (tons/hectare) focusing on Kerala rows.

Primary model: **XGBoost Regressor** (strong for tabular).

Datasets:
1. Kaggle: `akshatgupta7/crop-yield-in-indian-states-dataset`
2. data.gov.in: Crop Production Statistics (download separately)

Because public datasets may have gaps for Kerala districts/crops, the notebook supports **synthetic gap-filling** (clearly flagged).


In [ ]:
# Run this first in every notebook
!pip -q install scikit-learn xgboost pandas numpy matplotlib seaborn joblib kaggle

from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_PATH = '/content/drive/MyDrive/krishi_mitra_models/'
DATA_PATH = '/content/drive/MyDrive/krishi_mitra_data/'

import os
from pathlib import Path
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)


## 1) Download Kaggle dataset

Upload `kaggle.json` to Colab, then run.


In [ ]:
from pathlib import Path

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
if Path('kaggle.json').exists():
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json

raw_dir = Path(DATA_PATH) / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d akshatgupta7/crop-yield-in-indian-states-dataset -p "{raw_dir}" -q
!unzip -o "{raw_dir}/crop-yield-in-indian-states-dataset.zip" -d "{raw_dir}" > /dev/null

print('Raw dir:', raw_dir)
print('Files:', [p.name for p in raw_dir.glob('*.csv')][:10])


## 2) Load + normalize columns

Different sources use different column names. This notebook creates a standardized schema:

**Features**
- `crop_name`, `district`, `year`, `area_hectares`, `annual_rainfall_mm`, `avg_temperature`
- `soil_quality_index` (1–10), `fertilizer_kg_per_hectare`, `pesticide_kg_per_hectare`
- `irrigation_type`, `season`

**Target**
- `production_tons_per_hectare`


In [ ]:
import numpy as np
import pandas as pd

csvs = list(raw_dir.glob('*.csv'))
assert csvs, 'No CSVs found in raw_dir'

# Heuristic: pick the largest CSV
csv_path = max(csvs, key=lambda p: p.stat().st_size)
print('Using:', csv_path.name)
raw = pd.read_csv(csv_path)
raw.head()

In [ ]:
def normalize_yield_schema(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    cols = {c.lower().strip(): c for c in d.columns}

    def pick(*names):
        for n in names:
            if n in cols:
                return cols[n]
        return None

    state_col = pick('state_name', 'state')
    district_col = pick('district_name', 'district')
    crop_col = pick('crop', 'crop_name', 'cropname')
    year_col = pick('crop_year', 'year')
    area_col = pick('area', 'area_hectares', 'area_in_hectares')
    prod_col = pick('production', 'production_in_tonnes', 'production_tonnes')

    if state_col is None or crop_col is None or year_col is None or area_col is None or prod_col is None:
        raise ValueError('Could not map required columns. Please inspect raw columns and update mapping.')

    if district_col is None:
        d['district_name'] = 'Unknown'
        district_col = 'district_name'

    out = pd.DataFrame({
        'state': d[state_col].astype(str),
        'district': d[district_col].astype(str),
        'crop_name': d[crop_col].astype(str),
        'year': pd.to_numeric(d[year_col], errors='coerce'),
        'area_hectares': pd.to_numeric(d[area_col], errors='coerce'),
        'production_tonnes_total': pd.to_numeric(d[prod_col], errors='coerce'),
    })

    out['production_tons_per_hectare'] = out['production_tonnes_total'] / (out['area_hectares'] + 1e-6)

    return out

base = normalize_yield_schema(raw)
base = base[base['state'].str.lower().str.contains('kerala')].copy()
base = base.replace([np.inf, -np.inf], np.nan)
base = base.dropna(subset=['year','area_hectares','production_tons_per_hectare'])
base[['state','district','crop_name','year','area_hectares','production_tons_per_hectare']].head()

## 3) Add Kerala-friendly features + synthetic gap fill

Public yield datasets usually miss weather and input-use features. We add:
- rainfall & temperature (district priors + noise)
- fertilizer/pesticide intensity (coarse priors)
- irrigation type + season
- soil quality index (1–10)
- lag features (previous year yield, 3-year rolling average)
- drought_index + flood_risk_score

These are **synthetic proxies** to let you build a working pipeline; replace with real IMD/ICAR/KAU data when available.


In [ ]:
KERALA_DISTRICTS = [
    'Palakkad','Thrissur','Wayanad','Malappuram','Kannur',
    'Kozhikode','Idukki','Ernakulam','Alappuzha','Kottayam'
]

DISTRICT_PRIORS = {
    'Palakkad': {'temp': 29.0, 'rain': 1800},
    'Thrissur': {'temp': 28.0, 'rain': 2600},
    'Wayanad': {'temp': 24.5, 'rain': 3200},
    'Malappuram': {'temp': 27.5, 'rain': 2900},
    'Kannur': {'temp': 27.0, 'rain': 3300},
    'Kozhikode': {'temp': 27.0, 'rain': 3400},
    'Idukki': {'temp': 23.5, 'rain': 2800},
    'Ernakulam': {'temp': 28.0, 'rain': 3100},
    'Alappuzha': {'temp': 28.2, 'rain': 3000},
    'Kottayam': {'temp': 27.2, 'rain': 3200},
}

rng = np.random.default_rng(7)

def canonical_district(d: str) -> str:
    d2 = str(d).strip().title()
    # map unknowns to nearest prior bucket
    for k in DISTRICT_PRIORS:
        if k.lower() in d.lower():
            return k
    return rng.choice(list(DISTRICT_PRIORS.keys()))

data = base.copy()
data['district'] = data['district'].apply(canonical_district)

data['annual_rainfall_mm'] = data['district'].map(lambda d: DISTRICT_PRIORS[d]['rain']).astype(float)
data['annual_rainfall_mm'] = data['annual_rainfall_mm'] * rng.normal(1.0, 0.08, size=len(data))

data['avg_temperature'] = data['district'].map(lambda d: DISTRICT_PRIORS[d]['temp']).astype(float)
data['avg_temperature'] = data['avg_temperature'] + rng.normal(0, 1.2, size=len(data))

data['soil_quality_index'] = rng.integers(4, 9, size=len(data)).astype(float)
data['fertilizer_kg_per_hectare'] = rng.normal(140, 35, size=len(data)).clip(40, 280)
data['pesticide_kg_per_hectare'] = rng.normal(3.2, 1.1, size=len(data)).clip(0.2, 8.0)
data['irrigation_type'] = rng.choice(['Rainfed', 'Canal', 'Borewell', 'Pond'], size=len(data), p=[0.55, 0.2, 0.15, 0.1])
data['season'] = rng.choice(['Kharif', 'Rabi', 'Summer'], size=len(data), p=[0.55, 0.3, 0.15])

# Weather risk scores (simple proxies)
data['drought_index'] = (1.0 - (data['annual_rainfall_mm'] / 3500)).clip(0, 1)
data['flood_risk_score'] = ((data['annual_rainfall_mm'] / 3500) + (data['season'] == 'Kharif').astype(int) * 0.2).clip(0, 1)

# Lag features: previous year + rolling average
data = data.sort_values(['district', 'crop_name', 'year']).reset_index(drop=True)
data['previous_year_yield'] = data.groupby(['district', 'crop_name'])['production_tons_per_hectare'].shift(1)
data['avg_3yr_yield'] = data.groupby(['district', 'crop_name'])['production_tons_per_hectare'].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)

# Fill missing lag values with group medians
for col in ['previous_year_yield']:
    data[col] = data.groupby(['district','crop_name'])[col].transform(lambda s: s.fillna(s.median()))

data.head()

## 4) Data cleaning (missing + outliers)

- Fill missing numeric values with district-wise medians
- Remove outliers using IQR on the target


In [ ]:
num_cols = [
    'year','area_hectares','annual_rainfall_mm','avg_temperature','soil_quality_index',
    'fertilizer_kg_per_hectare','pesticide_kg_per_hectare','previous_year_yield','avg_3yr_yield',
    'drought_index','flood_risk_score'
]

for c in num_cols:
    data[c] = pd.to_numeric(data[c], errors='coerce')
    data[c] = data.groupby('district')[c].transform(lambda s: s.fillna(s.median()))

y = data['production_tons_per_hectare'].astype(float)
q1, q3 = y.quantile(0.25), y.quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
clean = data[(y >= lo) & (y <= hi)].copy()
print('Rows before:', len(data), 'after:', len(clean))

clean[['district','crop_name','year','area_hectares','production_tons_per_hectare']].head()

## 5) Train models (XGBoost / RandomForest / Linear)

We use a preprocessing pipeline with:
- OneHot for categorical
- StandardScaler for numeric


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from xgboost import XGBRegressor

feature_cols = [
    'crop_name','district','year','area_hectares','annual_rainfall_mm','avg_temperature','soil_quality_index',
    'fertilizer_kg_per_hectare','pesticide_kg_per_hectare','irrigation_type','season',
    'previous_year_yield','avg_3yr_yield','drought_index','flood_risk_score'
]
target_col = 'production_tons_per_hectare'

X = clean[feature_cols].copy()
y = clean[target_col].astype(float).copy()

cat_cols = ['crop_name','district','irrigation_type','season']
num_cols = [c for c in feature_cols if c not in cat_cols]

scaler = StandardScaler()
pre = ColumnTransformer([
    ('num', scaler, num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

candidates = {
    'XGBoost': XGBRegressor(
        n_estimators=900,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=600,
        random_state=42,
        n_jobs=-1,
    ),
    'LinearRegression': LinearRegression(),
}

metrics = []
fitted = {}
for name, reg in candidates.items():
    pipe = Pipeline([('pre', pre), ('reg', reg)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred, squared=False)
    r2 = r2_score(y_test, pred)
    metrics.append((name, mae, rmse, r2))
    fitted[name] = pipe

pd.DataFrame(metrics, columns=['model','MAE','RMSE','R2']).sort_values('RMSE')

## 6) Explainability

We plot feature importance for the best tree model (XGBoost/RandomForest).

SHAP values are optional (can be heavy in Colab); enable if needed.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

best_name = pd.DataFrame(metrics, columns=['model','MAE','RMSE','R2']).sort_values('RMSE').iloc[0]['model']
best_model = fitted[best_name]
print('Selected best:', best_name)

reg = best_model.named_steps['reg']
if hasattr(reg, 'feature_importances_'):
    # Get feature names after preprocessing
    ohe = best_model.named_steps['pre'].named_transformers_['cat']
    cat_names = list(ohe.get_feature_names_out(cat_cols))
    feat_names = num_cols + cat_names

    importances = reg.feature_importances_
    idx = np.argsort(importances)[::-1][:20]
    plt.figure(figsize=(10, 6))
    plt.barh(np.array(feat_names)[idx][::-1], importances[idx][::-1])
    plt.title('Top feature importances')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importances not available for this model')


## 7) Save model artifacts

Saves:
- `yield_model.pkl` (full pipeline)
- `scaler.pkl` (numeric scaler; included for compatibility)


In [ ]:
import joblib
from pathlib import Path

local_dir = Path('ml_models/3_yield_predictor/model')
local_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, local_dir / 'yield_model.pkl')
joblib.dump(scaler, local_dir / 'scaler.pkl')

drive_dir = Path(MODEL_SAVE_PATH) / 'yield_predictor'
drive_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, drive_dir / 'yield_model.pkl')
joblib.dump(scaler, drive_dir / 'scaler.pkl')

print('Saved to local:', local_dir)
print('Saved to drive:', drive_dir)


## 8) `predict()` demo

We provide a structured output including a rough confidence band based on test residual spread.


In [ ]:
import numpy as np
import pandas as pd

# Residual-based uncertainty proxy
test_pred = best_model.predict(X_test)
resid = (y_test - test_pred)
sigma = float(np.std(resid))

def predict(farmer_input: dict):
    required = ['crop','district','rainfall_mm','temperature','soil_quality','fertilizer_used','irrigation_type','season','area_hectares']
    missing = [k for k in required if k not in farmer_input]
    if missing:
        raise ValueError(f"Missing keys: {missing}")

    row = {
        'crop_name': farmer_input['crop'],
        'district': farmer_input['district'],
        'year': farmer_input.get('year', 2025),
        'area_hectares': float(farmer_input['area_hectares']),
        'annual_rainfall_mm': float(farmer_input['rainfall_mm']),
        'avg_temperature': float(farmer_input['temperature']),
        'soil_quality_index': float(farmer_input['soil_quality']),
        'fertilizer_kg_per_hectare': float(farmer_input['fertilizer_used']),
        'pesticide_kg_per_hectare': float(farmer_input.get('pesticide_used', 3.0)),
        'irrigation_type': farmer_input['irrigation_type'],
        'season': farmer_input['season'],
        # lag features (unknown for a new farm): use district/crop medians from training
        'previous_year_yield': float(clean.groupby(['district','crop_name'])[target_col].median().get((farmer_input['district'], farmer_input['crop']), clean[target_col].median())),
        'avg_3yr_yield': float(clean.groupby(['district','crop_name'])[target_col].mean().get((farmer_input['district'], farmer_input['crop']), clean[target_col].mean())),
        'drought_index': float((1.0 - (float(farmer_input['rainfall_mm']) / 3500.0)).clip(0, 1)),
        'flood_risk_score': float(((float(farmer_input['rainfall_mm']) / 3500.0) + (1 if farmer_input['season']=='Kharif' else 0)*0.2).clip(0, 1)),
    }

    X_one = pd.DataFrame([row])[feature_cols]
    yhat = float(best_model.predict(X_one)[0])

    district_avg = float(clean[clean['district'] == farmer_input['district']][target_col].median()) if farmer_input['district'] in clean['district'].unique() else float(clean[target_col].median())
    delta = (yhat - district_avg) / (district_avg + 1e-6) * 100
    comparison = f"{delta:+.0f}% vs district median"

    tips = []
    if farmer_input['season'] == 'Kharif' and farmer_input['rainfall_mm'] > 3000:
        tips.append('Improve drainage to reduce flood stress')
    if farmer_input['fertilizer_used'] < 80:
        tips.append('Review balanced NPK schedule; consider split application')
    tips.append('Scout for pests/diseases weekly during monsoon')

    return {
        'predicted_yield_tons_per_hectare': yhat,
        'confidence_range': {'min': max(0.0, yhat - 1.0*sigma), 'max': yhat + 1.0*sigma},
        'district_average': district_avg,
        'comparison': comparison,
        'top_yield_factors': ['rainfall', 'temperature', 'soil_quality', 'fertilizer_intensity'],
        'improvement_tips': tips,
    }

test_farmer = {
    'crop': 'Paddy',
    'district': 'Palakkad',
    'rainfall_mm': 2800,
    'temperature': 28,
    'soil_quality': 7,
    'fertilizer_used': 140,
    'irrigation_type': 'Canal',
    'season': 'Kharif',
    'area_hectares': 1.5,
    'year': 2025,
}

predict(test_farmer)